In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
providers_df = spark.read.csv("abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.providers.csv", header=True, inferSchema=True)

display(providers_df)

In [0]:
provider_id=providers_df.dropDuplicates(["provider_id"])
display(providers_df)

In [0]:
from pyspark.sql.functions import concat_ws, initcap, trim
full_name = providers_df.withColumn("full_name",concat_ws(" ", initcap(trim("first_name")), initcap(trim("last_name")))
)
display(full_name)

In [0]:
from pyspark.sql.functions import regexp_extract

npi_number_valid = full_name.withColumn(
    "npi_number_valid",
    regexp_extract("npi_number", r"^NPI(\d{10})$", 0)
)
display(npi_number_valid)

In [0]:

mci_number_clean = npi_number_valid.withColumn("mci_number_clean", trim("mci_number"))

display(mci_number_clean)


In [0]:
from pyspark.sql.functions import when, col

mci_unknown = mci_number_clean.withColumn(
    "mci_number",
    when(col("mci_number").isNull(), "UNKNOWN")
    .otherwise(col("mci_number"))
)
display(mci_unknown)

In [0]:
from pyspark.sql.functions import upper

designation_std = mci_unknown.withColumn("designation_std", upper(trim("designation")))
display(designation_std)

In [0]:
specialty_std = designation_std.withColumn("specialty_std", initcap(trim("specialty")))
display(specialty_std)
sub_specialty_std = specialty_std.withColumn("sub_specialty_std", initcap(trim("sub_specialty")))
display(sub_specialty_std)

In [0]:
from pyspark.sql.functions import when, col

sub_specialty_unknown = sub_specialty_std.withColumn(
    "sub_specialty",
    when(col("sub_specialty").isNull(), "UNKNOWN")
    .otherwise(col("sub_specialty"))
)
display(sub_specialty_unknown)


In [0]:
department_std = sub_specialty_unknown.withColumn("department_std", initcap(trim("department")))
display(department_std)

In [0]:
provider_type_std = department_std.withColumn("provider_type_std", initcap(trim("provider_type")))
display(provider_type_std)

In [0]:
from pyspark.sql.functions import col, when

exp_valid = provider_type_std.withColumn(
    "exp_valid",
    when(
        col("years_of_experience").cast("int").between(0, 60),
        col("years_of_experience").cast("int")
    ).otherwise(0)
)
display(exp_valid)

In [0]:
experience_band = exp_valid.withColumn(
    "experience_band",
    when(col("exp_valid").between(0, 5), "Junior")
    .when(col("exp_valid").between(6, 15), "Mid-Level")
    .when(col("exp_valid").between(16, 30), "Senior")
    .otherwise("Expert")
)
display(experience_band)

In [0]:
from pyspark.sql.functions import expr, when, col

consult_fee_valid = experience_band.withColumn(
    "consult_fee_valid",
    when(
        expr("try_cast(consultation_fee as double)") > 0,
        expr("try_cast(consultation_fee as double)")
    )
)
display(consult_fee_valid)

In [0]:
email_valid = consult_fee_valid.withColumn(
    "email_valid",
    when(col("email").rlike(r"^[^@]+@[^@]+\.[^@]+$"), col("email"))
)
display(email_valid)


In [0]:
from pyspark.sql.functions import col, when, upper, trim

is_active_flag = email_valid.withColumn(
    "is_active_flag",
    when(
        upper(trim(col("is_active").cast("string"))).isin("Y","YES","1","TRUE"),
        1
    ).when(
        upper(trim(col("is_active").cast("string"))).isin("N","NO","0","FALSE","INACTIVE"),
        0
    ).otherwise(None)
)
display(is_active_flag)

In [0]:
from pyspark.sql.functions import expr

facility_id_valid = is_active_flag.withColumn(
    "facility_id_valid",
    expr("try_cast(trim(facility_id) as bigint)")
)

display(facility_id_valid)

In [0]:
from pyspark.sql.functions import to_date

ingestion_date = facility_id_valid.withColumn("ingestion_date", to_date("ingestion_timestamp", "M/d/yy H:mm"))
display(ingestion_date)

In [0]:
is_specialist = ingestion_date.withColumn(
    "is_specialist",
    when(col("sub_specialty_std").isNotNull(), 1).otherwise(0)
)
display(is_specialist)


In [0]:
dq_provider_flag = is_specialist.withColumn(
    "dq_provider_flag",
    when(
        col("npi_number_valid").isNull() |
        (col("specialty_std") == "Unknown"),
        1
    ).otherwise(0)
)
display(dq_provider_flag)

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "providers"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
dq_provider_flag.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")
    